# Paper-Inspired Task A Model

**Based on:** Cotroneo, Improta, Liguori (2025) — *Human-Written vs. AI-Generated Code: A Large-Scale Study of Defects, Vulnerabilities, and Complexity* (arXiv:2508.21634)

## Architecture

- **Backbone:** `microsoft/codebert-base` (768-d [CLS] embedding)
- **Feature Vector:** 14 paper-inspired static features → `nn.LayerNorm(14)`
- **Fusion Head:** `[CLS](768) ⊕ features(14) = 782 → 256 → 128 → 64 → 2` (GELU + LayerNorm + Dropout)
- **Loss:** Weighted Focal Loss (γ=2, class weights from train distribution) to fix minority-class collapse

## 14 Paper-Inspired Features

Directly motivated by the paper's RQ3 (structural complexity) and RQ1 (defect profiles):

1. **NLOC** – lines of code (normalised) — paper finding: AI code is 6.75 lines shorter
2. **CCN** – cyclomatic complexity (normalised) — paper finding: AI is 1.66 simpler
3. **Token count (TC)** – total tokens (normalised) — paper finding: AI uses 63.74 fewer tokens
4. **Unique token ratio (UT/TC)** – lexical diversity — paper finding: AI has lower vocabulary
5. **Unique token count (UT)** – absolute vocabulary size (normalised)
6. **Function name length** – char count — paper finding: AI uses longer descriptive names
7. **Parameter count** – number of function parameters (normalised)
8. **Comment / docstring ratio** — paper: AI produces more verbose documentation patterns
9. **Blank line ratio** — paper: structural spacing differs
10. **Debug print density** — paper RQ1: AI overuses print/System.out.println (CWE-489)
11. **Subprocess/OS command presence** — paper RQ2: AI triggers CWE-78 far more (binary)
12. **Hardcoded credential heuristic** — paper RQ2: CWE-798 AI-dominant (binary)
13. **Exception handling ratio** — paper RQ2: humans trigger CWE-754 far more
14. **Line length CV** — coefficient of variation: AI more uniform, humans more varied

## Compute: Kaggle T4 (16 GB VRAM)

- `SAMPLE_SIZE = 100_000` (stratified across language + domain)
- `per_device_train_batch_size=8`, `gradient_accumulation_steps=4` → effective batch 32
- `fp16=True`, `MAX_LENGTH=512`
- LLRD + cosine warmup scheduler
- Early stopping (patience=3, metric=macro_f1)

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install --quiet --upgrade transformers==4.45.0 huggingface_hub lizard tiktoken
print("Dependencies ready.")

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import os
import re
import math
import hashlib
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from tqdm import tqdm

import lizard
import tiktoken

from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    get_cosine_schedule_with_warmup,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)

os.environ["WANDB_DISABLED"] = "true"
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# tiktoken encoder (GPT-4 schema; language-agnostic)
_ENC = tiktoken.get_encoding("cl100k_base")
print("All imports ready.")

## 1 · Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_ROOT = (
    "/kaggle/input/datasets/daniilor/semeval-2026-task13/"
    "SemEval-2026-Task13/task_a/"
)
TRAIN_PARQUET = DATA_ROOT + "task_a_training_set_1.parquet"
VAL_PARQUET   = DATA_ROOT + "task_a_validation_set.parquet"
TEST_PARQUET  = DATA_ROOT + "task_a_test_set_sample.parquet"
OUTPUT_DIR    = "/kaggle/working/paper_inspired_model"

# ── Hyperparameters ───────────────────────────────────────────────────────────
MODEL_NAME       = "microsoft/codebert-base"
MAX_LENGTH       = 512
SAMPLE_SIZE      = 100_000   # 100k stratified across language + domain
VAL_SAMPLE_SIZE  = 20_000    # 20k validation
RANDOM_SEED      = 42

NUM_EPOCHS       = 8         # early stopping will cut this short
BATCH_SIZE       = 8         # per-device; T4 safe
GRAD_ACCUM       = 4         # effective batch = 32
BASE_LR          = 2e-5
HEAD_LR          = 5e-4
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.06
GRAD_CLIP        = 1.0
LLRD_FACTOR      = 0.9       # slightly more aggressive LR decay for 12 layers
DROPOUT          = 0.2
FOCAL_GAMMA      = 2.0       # focal loss exponent

NUM_CODE_FEATURES = 14

print(f"Sample size  : {SAMPLE_SIZE:,}")
print(f"Val size     : {VAL_SAMPLE_SIZE:,}")
print(f"Effective BSZ: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Features     : {NUM_CODE_FEATURES}")

## 2 · Data Loading & Cleaning

In [ ]:
# ── Cleaning pipeline ────────────────────────────────────────────────────────
PLACEHOLDERS = {"unk", "na", "n/a", "-", "?", "none", "null", "nan"}
ENCODING_RE  = re.compile(r"Ã.|â\u20ac™|â\u20acœ|â\u20ac|\ufffd|\?\?\?")

def normalize_ws(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

def sha1_series(s: pd.Series) -> pd.Series:
    return s.map(lambda x: hashlib.sha1(x.encode("utf-8", "ignore")).hexdigest())

def clean_df(df: pd.DataFrame, text_col="code", label_col="label") -> pd.DataFrame:
    n0 = len(df)
    raw = df[text_col]
    missing = (raw.isna() | raw.fillna("").astype(str).eq("") |
               raw.fillna("").astype(str).str.fullmatch(r"\s*") |
               raw.fillna("").astype(str).str.strip().str.lower().isin(PLACEHOLDERS))
    df = df[~missing].reset_index(drop=True)

    enc_mask = normalize_ws(df[text_col]).str.contains(ENCODING_RE)
    df = df[~enc_mask].reset_index(drop=True)

    t = normalize_ws(df[text_col])
    h = sha1_series(t)
    df = df[~h.duplicated(keep="first")].reset_index(drop=True)

    t = normalize_ws(df[text_col])
    h = sha1_series(t)
    grp = pd.DataFrame({"h": h, "y": df[label_col].astype(str)})
    bad = set(grp.groupby("h")["y"].nunique().pipe(lambda s: s[s > 1]).index)
    df = df[~h.isin(bad)].reset_index(drop=True)

    print(f"  Cleaned: {n0:,} → {len(df):,} rows")
    return df


# ── Load & stratified sample ─────────────────────────────────────────────────
print("Loading training data...")
raw_df = pd.read_parquet(TRAIN_PARQUET)
raw_df["label"] = raw_df["label"].astype(int)
print(f"  Raw: {len(raw_df):,} rows | cols: {raw_df.columns.tolist()}")

train_full = clean_df(raw_df)

# Detect language and domain columns for stratified sampling
lang_col   = next((c for c in train_full.columns if c.lower() in ("language", "lang", "programming_language")), None)
domain_col = next((c for c in train_full.columns if c.lower() in ("domain", "task_type", "category")), None)

print(f"  Language col : {lang_col}")
print(f"  Domain col   : {domain_col}")
if lang_col:
    print(f"  Languages    : {train_full[lang_col].value_counts().to_dict()}")
if domain_col:
    print(f"  Domains      : {train_full[domain_col].value_counts().to_dict()}")

In [ ]:
# ── Stratified sample across label × language × domain ───────────────────────
# If we only have label, fall back to label-stratified sampling.
# Strategy: sample proportionally per (label, lang, domain) stratum to maximise
# diversity over unseen-language and unseen-domain evaluation settings.

def stratified_sample(df, n, seed, group_cols):
    """Proportional stratified sample. Falls back gracefully if cols missing."""
    cols = [c for c in group_cols if c in df.columns]
    if not cols:
        return df.sample(n=min(n, len(df)), random_state=seed).reset_index(drop=True)

    strat_key = df[cols].astype(str).agg("_".join, axis=1)
    counts = strat_key.value_counts()
    total  = len(df)
    samples = []
    remaining = n
    for key, cnt in counts.items():
        take = max(1, round(cnt / total * n))
        take = min(take, cnt)
        samples.append(df[strat_key == key].sample(n=take, random_state=seed))
        remaining -= take

    result = pd.concat(samples).reset_index(drop=True)
    # top-up or trim to exact n
    if len(result) < n:
        extra = df[~df.index.isin(result.index)].sample(
            n=min(n - len(result), len(df) - len(result)), random_state=seed
        )
        result = pd.concat([result, extra]).reset_index(drop=True)
    return result.head(n)


train_df = stratified_sample(
    train_full, SAMPLE_SIZE, RANDOM_SEED,
    group_cols=["label", lang_col, domain_col]
)
print(f"\nTrain sample: {len(train_df):,} rows")
print(f"  Label distribution:\n{train_df['label'].value_counts().sort_index()}")
if lang_col:
    print(f"  Language distribution:\n{train_df[lang_col].value_counts().head(10)}")
if domain_col:
    print(f"  Domain distribution:\n{train_df[domain_col].value_counts()}")

# ── Validation set ────────────────────────────────────────────────────────────
print("\nLoading validation data...")
val_raw = pd.read_parquet(VAL_PARQUET)
val_raw["label"] = val_raw["label"].astype(int)
val_clean = clean_df(val_raw)
val_df = stratified_sample(
    val_clean, VAL_SAMPLE_SIZE, RANDOM_SEED,
    group_cols=["label", lang_col, domain_col]
)
print(f"Val sample  : {len(val_df):,} rows")
print(f"  Label distribution:\n{val_df['label'].value_counts().sort_index()}")

## 3 · Paper-Inspired Feature Extraction (14 features)

All features are motivated by specific findings in the paper:
- **RQ3** (structural complexity): NLOC, CCN, TC, UT, UT/TC, FNL, param count
- **RQ1** (defect profiles): debug print density, exception ratio, comment ratio, line-length CV
- **RQ2** (security vulnerabilities CWE): subprocess presence, hardcoded credential heuristic

In [ ]:
# ── Regex patterns for lightweight defect heuristics ─────────────────────────
# Paper RQ2: AI code disproportionately triggers these CWE categories
_RE_DEBUG_PRINT = re.compile(
    r"\bprint\s*\("
    r"|System\.out\.print"
    r"|console\.log\s*\("
    r"|fmt\.Print"
    r"|\bprintln!\(",
    re.IGNORECASE
)
_RE_SUBPROCESS = re.compile(
    r"subprocess\."
    r"|os\.system\s*\("
    r"|exec\.Command"
    r"|Runtime\.getRuntime\(\)"
    r"|ProcessBuilder"
    r"|shell_exec\s*\("
    r"|popen\s*\(",
    re.IGNORECASE
)
_RE_HARDCODED = re.compile(
    r'(?:password|passwd|secret|api_key|token|credential|auth)\s*=\s*[\'"][^\'"]{4,}[\'"]',
    re.IGNORECASE
)
_RE_EXCEPTION = re.compile(
    r"\btry\b|\bcatch\b|\bexcept\b|\bfinally\b",
    re.IGNORECASE
)
_RE_COMMENT = re.compile(
    r"^\s*(#|//|/\*|\*)",
    re.MULTILINE
)


def extract_paper_features(code: str) -> list:
    """
    Extract 14 paper-motivated features.
    All values normalised to roughly [0, 1].

    Feature vector:
      [0]  NLOC (normalised by 50)
      [1]  CCN  (normalised by 10)
      [2]  Token count TC (normalised by 200)
      [3]  Unique token ratio UT/TC
      [4]  Unique token count UT (normalised by 300)
      [5]  Function name length FNL (normalised by 30)
      [6]  Parameter count (normalised by 10)
      [7]  Comment / docstring ratio
      [8]  Blank line ratio
      [9]  Debug print density (prints per NLOC, capped at 1)
      [10] Subprocess/OS-command presence (binary)
      [11] Hardcoded credential heuristic (binary)
      [12] Exception handling ratio (try-blocks per NLOC, capped at 1)
      [13] Line length coefficient of variation
    """
    if not code or not code.strip():
        return [0.0] * 14

    lines = code.split("\n")
    non_empty = [l for l in lines if l.strip()]
    total_lines = max(len(lines), 1)
    nloc = max(len(non_empty), 1)

    # ── lizard structural metrics ────────────────────────────────────────────
    try:
        result = lizard.analyze_file.analyze_source_code("_.py", code)
        if result.function_list:
            fn = result.function_list[0]  # analyse first function
            ccn     = min(fn.cyclomatic_complexity / 10.0, 1.0)
            fnl     = min(len(fn.name) / 30.0, 1.0)
            params  = min(fn.parameter_count / 10.0, 1.0)
        else:
            ccn = min(result.average_cyclomatic_complexity / 10.0, 1.0) if hasattr(result, 'average_cyclomatic_complexity') else 0.1
            fnl    = 0.0
            params = 0.0
    except Exception:
        ccn, fnl, params = 0.1, 0.0, 0.0

    # ── tiktoken lexical metrics ─────────────────────────────────────────────
    try:
        tokens = _ENC.encode(code)
        tc = len(tokens)
        ut = len(set(tokens))
    except Exception:
        tc, ut = 1, 1
    tc_norm  = min(tc / 200.0, 1.0)
    ut_norm  = min(ut / 300.0, 1.0)
    ut_ratio = ut / max(tc, 1)  # already in [0, 1]

    # ── Paper RQ2 heuristics (lightweight regex) ─────────────────────────────
    debug_count    = len(_RE_DEBUG_PRINT.findall(code))
    debug_density  = min(debug_count / max(nloc, 1), 1.0)
    subprocess_bin = float(bool(_RE_SUBPROCESS.search(code)))
    hardcoded_bin  = float(bool(_RE_HARDCODED.search(code)))
    except_count   = len(_RE_EXCEPTION.findall(code))
    except_ratio   = min(except_count / max(nloc, 1), 1.0)

    # ── Structural / stylometric ─────────────────────────────────────────────
    comment_lines  = len(_RE_COMMENT.findall(code))
    comment_ratio  = comment_lines / total_lines
    blank_ratio    = sum(1 for l in lines if not l.strip()) / total_lines

    lengths = [len(l) for l in non_empty]
    mean_len = sum(lengths) / len(lengths) if lengths else 0.0
    if len(lengths) > 1 and mean_len > 0:
        std_len = math.sqrt(sum((x - mean_len) ** 2 for x in lengths) / len(lengths))
        cv = min(std_len / mean_len / 2.0, 1.0)
    else:
        cv = 0.0

    nloc_norm = min(nloc / 50.0, 1.0)

    return [
        nloc_norm,        # [0]  NLOC
        ccn,              # [1]  CCN
        tc_norm,          # [2]  TC
        ut_ratio,         # [3]  UT/TC (lexical diversity)
        ut_norm,          # [4]  UT
        fnl,              # [5]  FNL
        params,           # [6]  param count
        comment_ratio,    # [7]  comment ratio
        blank_ratio,      # [8]  blank line ratio
        debug_density,    # [9]  debug print density (RQ1/RQ2)
        subprocess_bin,   # [10] subprocess presence (CWE-78)
        hardcoded_bin,    # [11] hardcoded cred (CWE-798)
        except_ratio,     # [12] exception handling (CWE-754 inverse)
        cv,               # [13] line-length CV
    ]


# ── Sanity check a few samples ─────────────────────────────────────────────
sample_code = train_df["code"].iloc[0]
fv = extract_paper_features(sample_code)
print(f"Feature vector length: {len(fv)}")
print(f"Sample feature values: {[round(v, 4) for v in fv]}")
assert len(fv) == NUM_CODE_FEATURES, f"Expected {NUM_CODE_FEATURES} features, got {len(fv)}"

In [ ]:
# ── Batch feature extraction with progress bar ────────────────────────────────
print("Extracting features for train set...")
train_features = [
    extract_paper_features(c)
    for c in tqdm(train_df["code"].tolist(), desc="Train features")
]
train_df = train_df.copy()
train_df["code_features"] = train_features

print("Extracting features for val set...")
val_features = [
    extract_paper_features(c)
    for c in tqdm(val_df["code"].tolist(), desc="Val features")
]
val_df = val_df.copy()
val_df["code_features"] = val_features

# Feature sanity: check for NaNs
import numpy as np
train_feat_arr = np.array(train_features)
assert not np.isnan(train_feat_arr).any(), "NaN found in train features!"
print(f"Feature stats (train):")
print(f"  Mean per dim: {train_feat_arr.mean(axis=0).round(4)}")
print(f"  Std  per dim: {train_feat_arr.std(axis=0).round(4)}")
print("Feature extraction complete ✓")

## 4 · Model Architecture

**Paper-CodeBERT Hybrid:**
```
CodeBERT [CLS] (768) ──────────────────────────────────┐
                                                        ├─ cat → 782
Paper Features (14) → LayerNorm(14) ───────────────────┘
    ↓
 Linear(782, 256) → LayerNorm(256) → GELU → Dropout(0.2)
    ↓
 Linear(256, 128) → LayerNorm(128) → GELU → Dropout(0.2)
    ↓
 Linear(128, 64)  → LayerNorm(64)  → GELU → Dropout(0.2)
    ↓
 Linear(64, 2)  → logits (Weighted Focal Loss)
```

In [ ]:
# ── Weighted Focal Loss ───────────────────────────────────────────────────────
class WeightedFocalLoss(nn.Module):
    """
    Focal Loss with per-class weights.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    Fixes minority-class collapse by:
    1. Weighting: down-weights easy majority-class examples
    2. Focal term (1-p)^gamma: focuses on hard, misclassified examples
    """
    def __init__(self, class_weights: torch.Tensor, gamma: float = 2.0):
        super().__init__()
        self.register_buffer("class_weights", class_weights)
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        # Standard weighted CE as base
        ce_loss = F.cross_entropy(
            logits, labels,
            weight=self.class_weights.to(logits.device),
            reduction="none"
        )
        # Focal modulation: (1 - p_t)^gamma
        p_t = torch.exp(-ce_loss)
        focal_loss = ((1 - p_t) ** self.gamma) * ce_loss
        return focal_loss.mean()


def compute_class_weights(df: pd.DataFrame, label_col="label") -> torch.Tensor:
    """Inverse-frequency class weights, normalised so mean ≈ 1."""
    counts = df[label_col].value_counts().sort_index()
    total = counts.sum()
    weights = total / (len(counts) * counts)
    w_tensor = torch.tensor(weights.values, dtype=torch.float32)
    print(f"Class weights: {dict(zip(counts.index, w_tensor.numpy().round(3)))}")
    return w_tensor


class_weights = compute_class_weights(train_df)


# ── Model ─────────────────────────────────────────────────────────────────────
class PaperInspiredCodeBERT(nn.Module):
    """
    CodeBERT [CLS] (768) ⊕ Paper-features (14) → Deep MLP head → 2 classes
    Uses Weighted Focal Loss to prevent minority-class collapse.
    """

    def __init__(
        self,
        model_name: str = "microsoft/codebert-base",
        num_labels: int = 2,
        num_features: int = NUM_CODE_FEATURES,
        dropout: float = DROPOUT,
        class_weights: torch.Tensor = None,
        gamma: float = FOCAL_GAMMA,
    ):
        super().__init__()
        self.num_labels = num_labels

        # ── Transformer backbone ───────────────────────────────────────────
        self.transformer = RobertaModel.from_pretrained(model_name)
        self.config = self.transformer.config
        hidden_size = self.config.hidden_size  # 768

        # ── Paper-feature normalisation ────────────────────────────────────
        self.feat_norm = nn.LayerNorm(num_features)

        # ── Deep fusion head: (768+14) → 256 → 128 → 64 → 2 ──────────────
        input_dim = hidden_size + num_features  # 782
        self.head = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(64, num_labels),
        )
        # Xavier init on head
        for m in self.head:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

        # ── Loss ───────────────────────────────────────────────────────────
        if class_weights is not None:
            self.loss_fn = WeightedFocalLoss(class_weights, gamma=gamma)
        else:
            self.loss_fn = WeightedFocalLoss(
                torch.ones(num_labels), gamma=gamma
            )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        code_features=None,
        **kwargs,
    ):
        out = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        cls_vec = out.last_hidden_state[:, 0, :]  # (B, 768)

        if code_features is not None:
            feat = self.feat_norm(code_features.float())
        else:
            feat = cls_vec.new_zeros(cls_vec.size(0), self.feat_norm.normalized_shape[0])

        combined = torch.cat([cls_vec, feat], dim=-1)  # (B, 782)
        logits = self.head(combined)

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)


print("PaperInspiredCodeBERT defined.")
print(f"  Architecture: [CLS](768) ⊕ paper_features({NUM_CODE_FEATURES}) → 256 → 128 → 64 → 2")
print(f"  Loss: Weighted Focal Loss (gamma={FOCAL_GAMMA})")

## 5 · Dataset & Collator

In [ ]:
# ── Tokeniser ──────────────────────────────────────────────────────────────
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples["code"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

def make_hf_dataset(df: pd.DataFrame) -> Dataset:
    """
    Build a HuggingFace Dataset from a DataFrame that already has
    a 'code_features' column (list of floats).
    """
    cols = ["code", "label", "code_features"]
    ds = Dataset.from_pandas(df[cols].reset_index(drop=True))
    ds = ds.map(tokenize_fn, batched=True, remove_columns=["code"],
                desc="Tokenising")
    ds = ds.rename_column("label", "labels")
    return ds


print("Building HF datasets...")
train_dataset = make_hf_dataset(train_df)
val_dataset   = make_hf_dataset(val_df)
print(f"  Train: {len(train_dataset):,} | Val: {len(val_dataset):,}")


# ── Data Collator ─────────────────────────────────────────────────────────
class PaperFeaturesCollator:
    """DataCollatorWithPadding + stacking of paper code_features."""

    def __init__(self, tok):
        self.base = DataCollatorWithPadding(tokenizer=tok)

    def __call__(self, features):
        code_feats = None
        if "code_features" in features[0]:
            code_feats = [f.pop("code_features") for f in features]
        batch = self.base(features)
        if code_feats is not None:
            batch["code_features"] = torch.tensor(
                code_feats, dtype=torch.float32
            )
        return batch


data_collator = PaperFeaturesCollator(tokenizer)
print("Data collator ready.")

## 6 · Training

- **LLRD (Layer-wise LR Decay):** embeddings get lowest LR, head gets highest
- **Cosine Schedule with Warmup**
- **Focal Loss** to fix minority-class collapse
- **Early Stopping:** patience=3 on macro F1

In [ ]:
# ── Layer-wise LR decay groups ────────────────────────────────────────────
def get_llrd_params(model, base_lr, head_lr, wd, llrd):
    no_decay = {"bias", "LayerNorm.weight", "LayerNorm.bias"}
    params = []
    num_layers = model.config.num_hidden_layers  # 12 for codebert-base

    # Embeddings (lowest LR)
    emb_wd, emb_nowd = [], []
    for n, p in model.transformer.embeddings.named_parameters():
        (emb_wd if not any(nd in n for nd in no_decay) else emb_nowd).append(p)
    emb_lr = base_lr * (llrd ** num_layers)
    if emb_wd:   params.append({"params": emb_wd,   "lr": emb_lr, "weight_decay": wd})
    if emb_nowd: params.append({"params": emb_nowd, "lr": emb_lr, "weight_decay": 0.0})

    # Encoder layers (increasing LR from bottom to top)
    for i, layer in enumerate(model.transformer.encoder.layer):
        lr_i = base_lr * (llrd ** (num_layers - i))
        wd_p, nowd_p = [], []
        for n, p in layer.named_parameters():
            (wd_p if not any(nd in n for nd in no_decay) else nowd_p).append(p)
        if wd_p:   params.append({"params": wd_p,   "lr": lr_i, "weight_decay": wd})
        if nowd_p: params.append({"params": nowd_p, "lr": lr_i, "weight_decay": 0.0})

    # Head + feature norm (highest LR)
    head_wd, head_nowd = [], []
    for module in [model.head, model.feat_norm]:
        for n, p in module.named_parameters():
            (head_wd if not any(nd in n for nd in no_decay) else head_nowd).append(p)
    if head_wd:   params.append({"params": head_wd,   "lr": head_lr, "weight_decay": wd})
    if head_nowd: params.append({"params": head_nowd, "lr": head_lr, "weight_decay": 0.0})

    return params


# ── Custom Trainer: LLRD + cosine scheduler ──────────────────────────────
class PaperModelTrainer(Trainer):
    def __init__(self, *args, llrd=0.9, head_lr=5e-4, **kwargs):
        self._llrd    = llrd
        self._head_lr = head_lr
        super().__init__(*args, **kwargs)

    def create_optimizer_and_scheduler(self, num_training_steps):
        param_groups = get_llrd_params(
            self.model,
            base_lr=self.args.learning_rate,
            head_lr=self._head_lr,
            wd=self.args.weight_decay,
            llrd=self._llrd,
        )
        self.optimizer = torch.optim.AdamW(param_groups)
        warmup = int(num_training_steps * self.args.warmup_ratio)
        self.lr_scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=warmup,
            num_training_steps=num_training_steps,
        )

print("PaperModelTrainer (LLRD + cosine) defined.")

In [ ]:
# ── Metrics: Macro F1 (primary) + per-class breakdown ─────────────────────
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=1)
    acc   = accuracy_score(labels, preds)
    # macro F1 is the competition metric
    _, _, macro_f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    return {
        "accuracy": acc,
        "macro_f1": macro_f1,       # ← optimise for this
        "f1": weighted_f1,
    }


# ── Initialise model ───────────────────────────────────────────────────────
model = PaperInspiredCodeBERT(
    model_name=MODEL_NAME,
    num_labels=2,
    num_features=NUM_CODE_FEATURES,
    dropout=DROPOUT,
    class_weights=class_weights,
    gamma=FOCAL_GAMMA,
).to(DEVICE)

total  = sum(p.numel() for p in model.parameters())
train_ = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model params: {total:,} total | {train_:,} trainable")


# ── TrainingArguments ─────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,    # effective BSZ = 32
    weight_decay=WEIGHT_DECAY,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=1000,                           # ~every 1k optimizer steps
    save_strategy="steps",
    save_steps=1000,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",          # competition metric
    greater_is_better=True,
    remove_unused_columns=False,
    learning_rate=BASE_LR,
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=GRAD_CLIP,
    save_total_limit=2,
    report_to=[],
    fp16=True,                                 # T4 half-precision
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)


# ── Trainer ───────────────────────────────────────────────────────────────
trainer = PaperModelTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    llrd=LLRD_FACTOR,
    head_lr=HEAD_LR,
)

print("\n" + "="*60)
print("  TRAINING CONFIGURATION SUMMARY")
print("="*60)
print(f"  Model            : {MODEL_NAME}")
print(f"  Head             : 782 → 256 → 128 → 64 → 2 (GELU + LayerNorm)")
print(f"  Features         : {NUM_CODE_FEATURES} paper-inspired static features")
print(f"  Loss             : Weighted Focal Loss (gamma={FOCAL_GAMMA})")
print(f"  Class weights    : {class_weights.numpy().round(3)}")
print(f"  Train samples    : {len(train_dataset):,}")
print(f"  Val samples      : {len(val_dataset):,}")
print(f"  Epochs           : {NUM_EPOCHS} (early stop patience=3)")
print(f"  Effective BSZ    : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Base LR          : {BASE_LR}")
print(f"  Head LR          : {HEAD_LR}")
print(f"  LLRD factor      : {LLRD_FACTOR}")
print(f"  Dropout          : {DROPOUT}")
print(f"  fp16             : True")
print("="*60)
print("\nStarting training...")
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nBest model saved to {OUTPUT_DIR}")

## 7 · Evaluation — 4 Official Subtask Settings

In [ ]:
# ── Evaluation utilities ──────────────────────────────────────────────────
SEEN_LANGS     = {"c++", "cpp", "python", "java"}
UNSEEN_LANGS   = {"go", "php", "c#", "csharp", "c", "javascript", "js"}
SEEN_DOMAINS   = {"algorithmic"}
UNSEEN_DOMAINS = {"research", "production"}


@torch.no_grad()
def predict_on_df(
    model: nn.Module,
    tokenizer,
    df: pd.DataFrame,
    max_length: int = MAX_LENGTH,
    batch_size: int = 32,
    device: str = DEVICE,
) -> pd.DataFrame:
    """Run batch inference and return df with 'prediction' column."""
    model.to(device).eval()
    codes = df["code"].tolist()
    preds = []

    for i in tqdm(range(0, len(codes), batch_size), desc="Predicting"):
        batch_codes = codes[i : i + batch_size]
        enc = tokenizer(
            batch_codes, truncation=True, padding=True,
            max_length=max_length, return_tensors="pt",
        )
        feat = torch.tensor(
            [extract_paper_features(c) for c in batch_codes],
            dtype=torch.float32,
        )
        logits = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
            code_features=feat.to(device),
        ).logits
        preds.extend(logits.argmax(dim=-1).cpu().tolist())

    result = df.copy()
    result["prediction"] = preds
    return result


def evaluate_by_category(df: pd.DataFrame, tag: str = "Model"):
    """Print per-category metrics for the 4 official evaluation splits."""
    lang_col   = next((c for c in df.columns if c.lower() in ("language", "lang", "programming_language")), None)
    domain_col = next((c for c in df.columns if c.lower() in ("domain", "task_type", "category")), None)

    if "label" not in df.columns:
        print(f"[{tag}] No 'label' column — cannot evaluate.")
        return

    if lang_col is None or domain_col is None:
        print(f"[{tag}] Language/domain columns missing. Overall only:")
        print(classification_report(df["label"], df["prediction"], zero_division=0))
        return

    df = df.copy()
    df["_l"] = df[lang_col].str.strip().str.lower()
    df["_d"] = df[domain_col].str.strip().str.lower()

    settings = [
        ("(i)   Seen Lang & Seen Domain",    SEEN_LANGS,   SEEN_DOMAINS),
        ("(ii)  Unseen Lang & Seen Domain",   UNSEEN_LANGS, SEEN_DOMAINS),
        ("(iii) Seen Lang & Unseen Domain",   SEEN_LANGS,   UNSEEN_DOMAINS),
        ("(iv)  Unseen Lang & Unseen Domain", UNSEEN_LANGS, UNSEEN_DOMAINS),
    ]

    print(f"\n{'='*70}")
    print(f"  TEST RESULTS — {tag}")
    print(f"{'='*70}")

    all_macro_f1s = []
    for name, langs, domains in settings:
        mask = df["_l"].isin(langs) & df["_d"].isin(domains)
        sub  = df[mask]
        n    = len(sub)
        if n == 0:
            print(f"\n  {name}: ** no samples **")
            continue
        y_t, y_p = sub["label"].values, sub["prediction"].values
        acc = accuracy_score(y_t, y_p)
        _, _, macro_f1, _ = precision_recall_fscore_support(y_t, y_p, average="macro",    zero_division=0)
        _, _, wgt_f1,   _ = precision_recall_fscore_support(y_t, y_p, average="weighted", zero_division=0)
        all_macro_f1s.append(macro_f1)
        print(f"\n  {name}  (n={n})")
        print(f"    Accuracy={acc:.4f}  Macro-F1={macro_f1:.4f}  Weighted-F1={wgt_f1:.4f}")
        print(classification_report(y_t, y_p, zero_division=0))

    # Overall
    y_t_all = df["label"].values
    y_p_all = df["prediction"].values
    acc_all = accuracy_score(y_t_all, y_p_all)
    _, _, macro_f1_all, _ = precision_recall_fscore_support(y_t_all, y_p_all, average="macro",    zero_division=0)
    _, _, wgt_f1_all,   _ = precision_recall_fscore_support(y_t_all, y_p_all, average="weighted", zero_division=0)

    print(f"\n  OVERALL  (n={len(df)})")
    print(f"    Accuracy={acc_all:.4f}  Macro-F1={macro_f1_all:.4f}  Weighted-F1={wgt_f1_all:.4f}")
    if all_macro_f1s:
        print(f"    Avg Macro-F1 across settings: {np.mean(all_macro_f1s):.4f}")
    print("="*70)


print("Evaluation utilities defined.")

In [ ]:
# ── Run evaluation on official test set ───────────────────────────────────
import os

if os.path.exists(TEST_PARQUET):
    print(f"\nLoading official test set: {TEST_PARQUET}")
    test_df = pd.read_parquet(TEST_PARQUET)
    test_df = test_df.dropna(subset=["code"]).reset_index(drop=True)
    if "label" in test_df.columns:
        test_df["label"] = test_df["label"].astype(int)
    print(f"Official test size: {len(test_df):,} rows")
    print(f"Columns: {test_df.columns.tolist()}")

    test_results = predict_on_df(
        model, tokenizer, test_df,
        max_length=MAX_LENGTH, batch_size=32, device=DEVICE,
    )
    evaluate_by_category(test_results, tag="PaperInspiredCodeBERT")
else:
    print(f"Test parquet not found at: {TEST_PARQUET}")

## 8 · Generate Submission CSV

In [ ]:
# ── Generate submission if test has no labels ─────────────────────────────
SUBMISSION_PARQUET = (
    "/kaggle/input/datasets/daniilor/semeval-2026-task13/"
    "SemEval-2026-Task13/task_a/task_a_test_set_sample.parquet"
)

if os.path.exists(SUBMISSION_PARQUET):
    print("Generating submission file...")
    sub_df = pd.read_parquet(SUBMISSION_PARQUET)
    sub_df = sub_df.dropna(subset=["code"]).reset_index(drop=True)

    sub_results = predict_on_df(
        model, tokenizer, sub_df,
        max_length=MAX_LENGTH, batch_size=32, device=DEVICE,
    )

    sub_out = pd.DataFrame({
        "id": sub_results.index,
        "label": sub_results["prediction"],
    })
    submission_path = "/kaggle/working/submission_task_a.csv"
    sub_out.to_csv(submission_path, index=False)
    print(f"Submission saved: {submission_path}")
    print(f"Prediction distribution:\n{sub_results['prediction'].value_counts()}")  
else:
    print("No submission parquet found — skipping.")

In [ ]:
# ── Configuration summary ─────────────────────────────────────────────────
print("\n" + "="*60)
print("  FINAL CONFIGURATION SUMMARY")
print("="*60)
print(f"  Paper ref        : Cotroneo et al. 2025 (arXiv:2508.21634)")
print(f"  Model            : {MODEL_NAME}")
print(f"  Head             : [CLS](768) ⊕ features({NUM_CODE_FEATURES}) → 782 → 256 → 128 → 64 → 2")
print(f"  Loss             : Weighted Focal Loss (gamma={FOCAL_GAMMA})")
print(f"  Class weights    : {class_weights.numpy().round(3)}")
print(f"  Train samples    : {SAMPLE_SIZE:,} (stratified lang×domain×label)")
print(f"  Val samples      : {VAL_SAMPLE_SIZE:,}")
print(f"  MAX_LENGTH       : {MAX_LENGTH}")
print(f"  Effective BSZ    : {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Base LR          : {BASE_LR}")
print(f"  Head LR          : {HEAD_LR}")
print(f"  LLRD             : {LLRD_FACTOR}")
print(f"  Dropout          : {DROPOUT}")
print(f"  fp16             : True")
print(f"  Optimise for     : Macro F1 (task metric)")
print("="*60)
print("\n Done! ✓")